# Модуль 6 — LLM API на практике

Домашка к [лекции 6](https://itrubnikov.github.io/Train_of_Thought/docs/modules/06-llm-api/). Вы соберёте три утилиты на реальном API: summarizer (streaming), классификатор писем (structured output) и переводчик-редактор (многоходовый диалог). Нужен API-ключ — см. README.

## 0. Установка и клиент

Запустите ячейку. В Colab ключ берётся из Secrets (значок 🔑 слева, имя `ANTHROPIC_API_KEY`); локально — из `.env` (скопируйте `.env.example`).

In [ ]:
!pip -q install anthropic openai python-dotenv pydantic
import os

# ключ: Colab Secrets -> переменная окружения; иначе .env
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Нет ANTHROPIC_API_KEY. Впишите его в .env (см. .env.example) или в Colab Secrets.")

from anthropic import Anthropic
client = Anthropic()                 # читает ANTHROPIC_API_KEY из окружения
MODEL = "claude-haiku-4-5"           # дешёвый для учёбы; флагман — "claude-opus-4-8"
print("Готово, клиент создан.")

## 1. summarizer со streaming (TODO 1)

Реализуйте `summarize(text)`: вызов `client.messages.stream(...)`, печать токенов потоком из `stream.text_stream`, возврат `final.content[0].text`. Не забудьте `system`-промпт и `max_tokens`.

In [ ]:
ARTICLE = (
    "Трансформеры вытеснили рекуррентные сети, потому что механизм attention "
    "позволяет обрабатывать всю последовательность параллельно, а не по шагам. "
    "Это дало масштабируемость на GPU и способность ловить дальние связи в тексте. "
    "На этой архитектуре выросли все современные большие языковые модели."
)

def summarize(text: str) -> str:
    # TODO 1: client.messages.stream(model=MODEL, max_tokens=..., system=..., messages=[...])
    #         в цикле печатайте chunk из stream.text_stream (end="", flush=True),
    #         затем final = stream.get_final_message(); напечатайте final.usage;
    #         верните final.content[0].text
    raise NotImplementedError("TODO 1: реализуйте summarize со streaming")

print(summarize(ARTICLE))

## 2. Классификатор писем (TODO 2) — Build-twice

Сначала запустите наивную версию через `json.loads` и поймайте, где она ломается. Потом реализуйте `classify` через structured output — и сравните надёжность.

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel

EMAIL = "Здравствуйте! Не пришёл счёт за март, а оплатить надо сегодня. Помогите срочно."

# --- Заход 1: в лоб. Запустите и посмотрите, ломается ли (```json, лишний текст, выдуманная категория) ---
def classify_naive(email: str) -> dict:
    resp = client.messages.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "user", "content":
            f"Верни JSON с полями category, urgency, needs_reply для письма:\n\n{email}"}],
    )
    return json.loads(resp.content[0].text)   # может упасть на обёртке или лишнем тексте

# попробуйте несколько раз и запишите, что вышло:
# print(classify_naive(EMAIL))

# --- Заход 2: по-инженерному. structured output гарантирует схему ---
class EmailLabel(BaseModel):
    category: Literal["спам", "счёт", "поддержка", "личное", "другое"]
    urgency: Literal["низкая", "средняя", "высокая"]
    needs_reply: bool

def classify(email: str) -> EmailLabel:
    # TODO 2: client.messages.parse(model=MODEL, max_tokens=..., messages=[...], output_format=EmailLabel)
    #         верните resp.parsed_output (это уже EmailLabel)
    raise NotImplementedError("TODO 2: реализуйте classify через structured output")

print(classify(EMAIL))

## 3. Переводчик-редактор (TODO 3) — многоходовый диалог

API stateless: историю ведёте вы. Переведите фразу, затем по второй реплике получите правку. Покажите, что без добавления ответа модели в `messages` контекст теряется.

In [ ]:
SYSTEM = "Ты переводчик-редактор: переводишь на русский и улучшаешь стиль."

def translate_then_edit(phrase: str, instruction: str) -> str:
    messages = [{"role": "user", "content": f"Translate: {phrase!r}"}]
    # TODO 3: первый вызов client.messages.create(system=SYSTEM, messages=messages, ...);
    #         добавьте {"role": "assistant", "content": r1.content} в messages;
    #         добавьте {"role": "user", "content": instruction};
    #         второй вызов -> верните r2.content[0].text.
    #         Эксперимент: уберите строку с assistant и посмотрите, теряется ли контекст.
    raise NotImplementedError("TODO 3: реализуйте многоходовый диалог")

print(translate_then_edit("The cat sat on the mat.", "Сделай официальнее."))

## Что сдать

- [ ] TODO 1: `summarize` выдаёт ответ потоком, возвращает строку, печатает `usage`.
- [ ] TODO 2: записан пример, где наивный `json.loads` сломался; `classify` возвращает провалидированный `EmailLabel`.
- [ ] TODO 3: правка во второй реплике работает; показано, что без истории контекст теряется.
- [ ] Ключ не захардкожен (только `.env` / Secrets).

Вывод одной фразой запишите в ячейку ниже: что удивило больше всего.

_(Ваш вывод одной фразой здесь.)_